# Embed locally (M4 / MPS)

Run on your Mac after downloading `sample.parquet` + `catalog.parquet` from Kaggle into `artifacts/`.
Embeds the catalog with bge-large on Apple MPS and builds the FAISS index. One-time, cached.

`catalog.parquet` row order == `embeddings.npy` row order, so recommenders map `catalog['book_id']` rows directly to embedding rows.

In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import pandas as pd, faiss
from sentence_transformers import SentenceTransformer
from book_recsys.features.document import build_documents
from book_recsys.features.embed import embed_documents
from book_recsys.features.index import build_index

In [ ]:
ART = "artifacts"
catalog = pd.read_parquet(f"{ART}/catalog.parquet")
assert "author" in catalog.columns, (
    "catalog.parquet has no `author` column -> author would NOT be embedded. "
    "Re-run 02_preprocess cell 5 with the authors file mounted, then re-upload catalog.parquet.")
print("columns:", list(catalog.columns), "| shape:", catalog.shape)

In [ ]:
import torch
docs = build_documents(catalog)
print("sample document (must contain 'by <Author>'):\n", docs[0][:200])
# Auto-detect: CUDA (Windows/NVIDIA) -> MPS (Apple M-series) -> CPU.
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"embedding on {device}")
# On the 4GB Quadro M2200, lower batch_size if you hit CUDA OOM (e.g. encode_kwargs).
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
emb = embed_documents(docs, model, f"{ART}/embeddings.npy")
emb.shape

In [ ]:
index = build_index(emb)
faiss.write_index(index, f"{ART}/faiss.idx")
index.ntotal

Artifacts now in `artifacts/`: `sample.parquet`, `catalog.parquet`, `embeddings.npy`, `faiss.idx`. These feed the recommenders, eval, and the LLM retrieval stage (Plan 3).